## Setup Storage Access with OAuth

In [0]:


import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, timedelta
import logging

# COMMAND ----------

# MAGIC %md
# MAGIC ## OAuth Configuration for Azure Storage

# COMMAND ----------

# Storage credentials
STORAGE_ACCOUNT = "learningdb"
CONTAINER = "datalake"
APP_ID =""
TENANT_ID =""
SECRET = ""  # ⚠️ REPLACE with your actual secret VALUE (not Secret ID

# Configure OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net", APP_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net", SECRET)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net", f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token")

print("✅ OAuth configured successfully!")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Configuration Management

# COMMAND ----------

class Config:
    """Configuration manager for the project"""
    
    def __init__(self, storage_account, container):
        self.storage_account = storage_account
        self.container = container
        self.base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
        
    def get_source_path(self, table_name):
        return f"{self.base_path}/source/{table_name}"
    
    def get_bronze_path(self, table_name):
        return f"{self.base_path}/bronze/{table_name}"
    
    def get_silver_path(self, table_name):
        return f"{self.base_path}/silver/{table_name}"
    
    def get_gold_path(self, table_name):
        return f"{self.base_path}/gold/{table_name}"
    
    def get_checkpoint_path(self, layer, table_name):
        return f"{self.base_path}/checkpoint/{layer}/{table_name}"

# Initialize config with your storage account
config = Config(STORAGE_ACCOUNT, CONTAINER)
print(f"✅ Config initialized: {config.base_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Quality Checks

# COMMAND ----------

def check_null_percentage(df, column_name, threshold=0.1):
    """
    Check if null percentage exceeds threshold
    Returns: (passed, null_percentage)
    """
    total_count = df.count()
    null_count = df.filter(col(column_name).isNull()).count()
    null_pct = null_count / total_count if total_count > 0 else 0
    
    passed = null_pct <= threshold
    return passed, null_pct

def check_duplicate_count(df, key_columns):
    """
    Check for duplicate records based on key columns
    Returns: (passed, duplicate_count)
    """
    total_count = df.count()
    unique_count = df.dropDuplicates(key_columns).count()
    duplicate_count = total_count - unique_count
    
    passed = duplicate_count == 0
    return passed, duplicate_count

def check_schema_compliance(df, expected_schema):
    """
    Check if DataFrame schema matches expected schema
    Returns: (passed, differences)
    """
    actual_columns = set(df.columns)
    expected_columns = set(expected_schema.keys())
    
    missing_columns = expected_columns - actual_columns
    extra_columns = actual_columns - expected_columns
    
    passed = len(missing_columns) == 0 and len(extra_columns) == 0
    differences = {
        "missing": list(missing_columns),
        "extra": list(extra_columns)
    }
    
    return passed, differences

def validate_data_quality(df, table_name, quality_rules):
    """
    Run all quality checks and return summary
    """
    results = {
        "table": table_name,
        "timestamp": datetime.now().isoformat(),
        "total_records": df.count(),
        "checks": []
    }
    
    for rule in quality_rules:
        rule_type = rule["type"]
        
        if rule_type == "null_check":
            passed, value = check_null_percentage(df, rule["column"], rule.get("threshold", 0.1))
            results["checks"].append({
                "rule": f"Null check on {rule['column']}",
                "passed": passed,
                "value": f"{value:.2%}"
            })
        
        elif rule_type == "duplicate_check":
            passed, value = check_duplicate_count(df, rule["key_columns"])
            results["checks"].append({
                "rule": f"Duplicate check on {', '.join(rule['key_columns'])}",
                "passed": passed,
                "value": f"{value} duplicates"
            })
    
    results["overall_status"] = all(check["passed"] for check in results["checks"])
    return results

# COMMAND ----------

# MAGIC %md
# MAGIC ## Logging Helpers

# COMMAND ----------

class DataLogger:
    """Custom logger for data pipeline"""
    
    def __init__(self, notebook_name):
        self.notebook_name = notebook_name
        self.logs = []
    
    def info(self, message):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "level": "INFO",
            "notebook": self.notebook_name,
            "message": message
        }
        self.logs.append(log_entry)
        print(f"[INFO] {message}")
    
    def error(self, message, exception=None):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "level": "ERROR",
            "notebook": self.notebook_name,
            "message": message,
            "exception": str(exception) if exception else None
        }
        self.logs.append(log_entry)
        print(f"[ERROR] {message}")
        if exception:
            print(f"Exception: {exception}")
    
    def get_logs(self):
        return self.logs

# COMMAND ----------

# MAGIC %md
# MAGIC ## Helper Functions

# COMMAND ----------

def add_audit_columns(df, load_date=None):
    """Add audit columns to DataFrame"""
    if load_date is None:
        load_date = datetime.now()
    
    return df \
        .withColumn("created_date", lit(load_date)) \
        .withColumn("modified_date", lit(load_date)) \
        .withColumn("is_active", lit(True))

def get_incremental_data(df, date_column, days_back=7):
    """Filter data for incremental load"""
    cutoff_date = datetime.now() - timedelta(days=days_back)
    return df.filter(col(date_column) >= cutoff_date)

def deduplicate_records(df, key_columns, order_column="modified_date"):
    """Remove duplicates keeping the latest record"""
    from pyspark.sql.window import Window
    
    window_spec = Window.partitionBy(key_columns).orderBy(col(order_column).desc())
    
    return df \
        .withColumn("row_num", row_number().over(window_spec)) \
        .filter(col("row_num") == 1) \
        .drop("row_num")

def clean_string_columns(df):
    """Clean string columns - trim, remove special characters"""
    string_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]
    
    for col_name in string_columns:
        df = df.withColumn(col_name, trim(col(col_name)))
    
    return df

def write_delta_table(df, path, mode="append", partition_by=None, merge_keys=None):
    """
    Write DataFrame as Delta table with optional merge
    """
    if merge_keys:
        # Perform merge (upsert) operation
        from delta.tables import DeltaTable
        
        if DeltaTable.isDeltaTable(spark, path):
            delta_table = DeltaTable.forPath(spark, path)
            
            merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_keys])
            
            delta_table.alias("target") \
                .merge(df.alias("source"), merge_condition) \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()
        else:
            # First time write
            writer = df.write.format("delta").mode("overwrite")
            if partition_by:
                writer = writer.partitionBy(partition_by)
            writer.save(path)
    else:
        # Simple write
        writer = df.write.format("delta").mode(mode)
        if partition_by:
            writer = writer.partitionBy(partition_by)
        writer.save(path)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Profiling

# COMMAND ----------

def profile_dataframe(df, sample_size=10000):
    """
    Generate basic profiling statistics for a DataFrame
    """
    profile = {
        "total_records": df.count(),
        "total_columns": len(df.columns),
        "column_details": []
    }
    
    # Sample for faster processing
    sample_df = df.limit(sample_size) if df.count() > sample_size else df
    
    for column in df.columns:
        col_type = df.schema[column].dataType
        
        col_info = {
            "name": column,
            "type": str(col_type),
            "null_count": df.filter(col(column).isNull()).count(),
            "distinct_count": df.select(column).distinct().count()
        }
        
        # Numeric statistics
        if isinstance(col_type, (IntegerType, LongType, FloatType, DoubleType)):
            stats = sample_df.agg(
                min(column).alias("min"),
                max(column).alias("max"),
                avg(column).alias("avg")
            ).collect()[0]
            
            col_info["min"] = stats["min"]
            col_info["max"] = stats["max"]
            col_info["avg"] = stats["avg"]
        
        profile["column_details"].append(col_info)
    
    return profile

# COMMAND ----------

print("Utilities loaded successfully!")
